# TuneKit: Fine-Tune Phi-4 Mini

## Dataset Overview
| Metric | Value |
|--------|-------|
| **Model** | Phi-4 Mini (`microsoft/Phi-4-mini-instruct`) |
| **Examples** | 300 conversations |
| **Task Type** | Qa |
| **Estimated Time** | ~13 minutes |

## Before You Start
1. **GPU Required**: Go to `Runtime` > `Change runtime type` > Select **T4 GPU**
2. **Run All Cells**: Click `Runtime` > `Run all` or press `Ctrl+F9`
3. **Download Model**: After training, download your fine-tuned model from the Files panel

## Why Unsloth?
- **2-5x faster training** with optimized kernels
- **70% less memory** usage
- **Native 4-bit quantization** for efficiency
- **Easy export** to GGUF, 16-bit, 4-bit formats

---
*Generated by [TuneKit](https://github.com/riyanshibohra/TuneKit) - Fine-tuning made simple*


In [ ]:
%%capture
# Install Unsloth - 2x faster training, 70% less memory
!pip install unsloth


In [ ]:
# Verify GPU
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU")


✅ GPU: Tesla T4
   Memory: 14.6 GB


## HuggingFace Login (Optional)

If you want to push your model to HuggingFace Hub later, login now.
Otherwise, skip this cell.


In [ ]:
# Optional: Login to HuggingFace (for pushing models to Hub)
# Uncomment the lines below if you want to upload your model later

# from huggingface_hub import login
# login()  # Will prompt for your token

# Get your token at: https://huggingface.co/settings/tokens


In [ ]:
#@title Load Training Dataset (click to expand)
import base64
import json

# Your dataset (embedded from TuneKit)
DATASET_B64 = """..."""

# Decode and parse
dataset_jsonl = base64.b64decode(DATASET_B64).decode()
conversations = [json.loads(line) for line in dataset_jsonl.strip().split("\n") if line.strip()]

print(f"Loaded {len(conversations):,} conversations")
print(f"\nSample conversation:")
if conversations:
    sample = conversations[0]
    for msg in sample.get("messages", [])[:3]:
        role = msg.get("role", "unknown")
        content = msg.get("content", "")[:100]
        print(f"  {role}: {content}...")


Loaded 300 conversations

Sample conversation:
  system: You are a mathematical reasoning assistant. Always solve problems step by step. Keep reasoning conci...
  user: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How m...
  assistant: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether...


In [ ]:
# Load model with Unsloth (4-bit quantization for memory efficiency)
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"  # Disable dynamo to avoid compilation errors

from unsloth import FastLanguageModel
import torch

MODEL_ID = "microsoft/Phi-4-mini-instruct"
MAX_SEQ_LENGTH = 2048

print(f"Loading {MODEL_ID} with Unsloth...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect (float16 for T4)
    load_in_4bit=True,  # 4-bit quantization
)

# Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Rank - lower = smaller adapter, faster training
    lora_alpha=16,  # Scaling factor
    lora_dropout=0,  # No dropout for faster training
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "fc1", "fc2"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Optimized checkpointing (4x longer context)
    random_state=42,
)

# Print trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Model loaded!")
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading microsoft/Phi-4-mini-instruct with Unsloth...
==((====))==  Unsloth 2026.2.1: Fast Phi3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.23G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Unsloth: Making `model.base_model.model.model` require gradients
Model loaded!
Trainable parameters: 3,145,728 / 2,341,800,960 (0.13%)


In [ ]:
# Prepare dataset for training
from datasets import Dataset

def format_conversation(example):
    """Format conversation using chat template."""
    messages = example.get("messages", [])

    # Apply chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

# Create dataset
dataset = Dataset.from_list(conversations)
dataset = dataset.map(format_conversation, remove_columns=dataset.column_names)

print(f"Dataset prepared: {len(dataset)} examples")
print(f"\nSample formatted text (first 500 chars):")
print(dataset[0]["text"][:500] + "...")


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset prepared: 300 examples

Sample formatted text (first 500 chars):
<|system|>You are a mathematical reasoning assistant. Always solve problems step by step. Keep reasoning concise. End your response with the final numeric answer. After the final answer, output <STOP>.<|end|><|user|>Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|end|><|assistant|>Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April...


In [ ]:
# Train with SFTTrainer (optimized for Unsloth)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    output_dir="./results",

    # Training params
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    # Optimizer
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=5,

    # Precision (auto-detect best for GPU)
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    # Logging
    logging_steps=10,

    # Saving
    save_strategy="epoch",
    save_total_limit=2,

    # Optimization
    optim="adamw_8bit",  # 8-bit Adam for memory efficiency
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,  # Set True for short sequences
    args=training_args,
)

# GPU stats before training
gpu_stats = torch.cuda.get_device_properties(0)
start_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
max_memory = round(gpu_stats.total_memory / 1024**3, 2)
print(f"GPU: {gpu_stats.name}")
print(f"Memory before: {start_memory} GB / {max_memory} GB")

print("\nStarting training...")
print("=" * 50)

trainer_stats = trainer.train()

# GPU stats after training
end_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print("=" * 50)
print(f"Training complete!")
print(f"Peak GPU memory: {end_memory} GB ({round(end_memory/max_memory*100, 1)}% of {max_memory} GB)")


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/300 [00:00<?, ? examples/s]

GPU: Tesla T4
Memory before: 3.65 GB / 14.56 GB

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 3 | Total steps = 114
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 3,145,728 of 3,839,167,488 (0.08% trained)


Step,Training Loss
10,1.608900
20,0.997100
30,0.626700
40,0.543200
50,0.556000
60,0.546200
70,0.508100
80,0.521300
90,0.505400
100,0.471100


Training complete!
Peak GPU memory: 9.79 GB (67.2% of 14.56 GB)


In [ ]:
# Save the fine-tuned model
import os

OUTPUT_DIR = "./fine_tuned_phi_4_mini"

# Save LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"LoRA adapter saved to: {OUTPUT_DIR}")

# List saved files
files = os.listdir(OUTPUT_DIR)
print(f"\nFiles saved:")
for f in files:
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f"  {f} ({size:.1f} KB)")


LoRA adapter saved to: ./fine_tuned_phi_4_mini

Files saved:
  merges.txt (2361.7 KB)
  special_tokens_map.json (0.6 KB)
  adapter_model.safetensors (12296.3 KB)
  chat_template.jinja (0.4 KB)
  adapter_config.json (1.2 KB)
  vocab.json (3818.7 KB)
  added_tokens.json (0.3 KB)
  tokenizer.json (15160.6 KB)
  tokenizer_config.json (2.8 KB)
  README.md (5.1 KB)


In [ ]:
# Test your fine-tuned model
FastLanguageModel.for_inference(model)  # Enable faster inference

# System prompt from your training data
SYSTEM_PROMPT = """You are a mathematical reasoning assistant. Always solve problems step by step. Keep reasoning concise. End your response with the final numeric answer. After the final answer, output <STOP>."""

def generate_response(prompt, max_new_tokens=256):
    """Generate a response from the fine-tuned model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        use_cache=True,
    )

    # Decode only new tokens
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

# Test it!
test_prompts = [
    "Hello! Can you help me?",
    "What can you do?",
]

print("Testing fine-tuned model:")
print("=" * 50)
for prompt in test_prompts:
    print(f"\nUser: {prompt}")
    response = generate_response(prompt)
    print(f"Assistant: {response}")
    print("-" * 1050)


Testing fine-tuned model:

User: Hello! Can you help me?
Assistant: Of course! I’m here to help.
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

---
## RL Stage — Imports & Setup

In [ ]:
# Additional imports for RL extensions and evaluation
import re, math, time, random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Tuple, Optional
import torch
import torch.nn.functional as F

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


In [ ]:
# System prompt used across SFT, RL, and evaluation stages
SYSTEM_PROMPT = (
    'You are a mathematical reasoning assistant. Always solve problems step by step. '
    'Keep reasoning concise. End your response with the final numeric answer. '
    'After the final answer, output <STOP>.'
)
print('SYSTEM_PROMPT set')


In [ ]:
def extract_answer(text: str):
    """Extract final numeric answer from model output."""
    patterns = [
        r'####\s*([\-\d\.\,]+)',
        r'\*\*Answer:\s*([\-\d\.\,]+)',
        r'Answer:\s*([\-\d\.\,]+)',
        r'=\s*([\-\d\.\,]+)\s*(?:<STOP>)?\s*$',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.MULTILINE)
        if m:
            try:
                return str(float(m.group(1).replace(',', '')))
            except ValueError:
                pass
    nums = re.findall(r'[\-]?\d+\.?\d*', text)
    return str(float(nums[-1])) if nums else None


def is_correct(model_output: str, ground_truth: str) -> bool:
    pred = extract_answer(model_output)
    try:
        gt = str(float(ground_truth.replace(',', '')))
        return pred is not None and abs(float(pred) - float(gt)) < 1e-3
    except (ValueError, TypeError):
        return pred is not None and pred.strip() == ground_truth.strip()


print('Helper functions defined')


In [ ]:
class RewardModule:
    """
    R(x, y) = 1{y_correct} . (1 - alpha . sigmoid((len - mu_x) / sigma_x))
    Shortest correct answer -> highest reward. Wrong answers -> 0.
    """

    def __init__(self, alpha: float = 0.1):
        self.alpha = alpha

    @staticmethod
    def _sigmoid(x: float) -> float:
        return 1.0 / (1.0 + math.exp(-max(-500, min(500, x))))

    def compute_rewards(
        self,
        responses: List[str],
        ground_truth: str,
        alpha: Optional[float] = None,
    ) -> Tuple[List[float], List[bool]]:
        alpha       = alpha if alpha is not None else self.alpha
        correctness = [is_correct(r, ground_truth) for r in responses]
        lengths     = [len(r.split()) for r in responses]
        correct_lens = [l for l, c in zip(lengths, correctness) if c]
        if correct_lens:
            mu    = np.mean(correct_lens)
            sigma = max(np.std(correct_lens), 1.0)
        else:
            mu    = np.mean(lengths)
            sigma = max(np.std(lengths), 1.0)
        rewards = []
        for length, correct in zip(lengths, correctness):
            if not correct:
                rewards.append(0.0)
            else:
                z = (length - mu) / sigma
                f = self._sigmoid(z)
                rewards.append(float(np.clip(1.0 - alpha * f, 0.0, 1.0)))
        return rewards, correctness


print('RewardModule defined')


In [ ]:
class RLOOAdvantage:
    """
    A_i = R_i - (1/(n-1)) * sum(R_j for j != i)
    No separate critic -> 50-70% less GPU memory.
    """

    @staticmethod
    def compute(rewards: List[float]) -> List[float]:
        n          = len(rewards)
        assert n >= 2, 'Need at least 2 samples for RLOO'
        R          = np.array(rewards, dtype=np.float32)
        total      = R.sum()
        baselines  = (total - R) / (n - 1)
        advantages = R - baselines
        std = advantages.std()
        if std > 1e-8:
            advantages = advantages / std
        return advantages.tolist()


print('RLOOAdvantage defined')


In [ ]:
class PPOTrainerRLOO:
    def __init__(self, model, ref_model, tokenizer,
                 reward_module, alpha=0.1,
                 ppo_clip=0.2, kl_beta=0.02,
                 lr=1e-5, n_samples=8, max_new_tokens=256):
        self.model      = model
        self.ref_model  = ref_model
        self.tokenizer  = tokenizer
        self.reward     = reward_module
        self.alpha      = alpha
        self.eps        = ppo_clip
        self.beta       = kl_beta
        self.n          = n_samples
        self.max_new    = max_new_tokens
        self.optimizer  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        self.history    = defaultdict(list)

    @torch.no_grad()
    def _generate_one(self, input_ids):
        out = self.model.generate(
            input_ids,
            max_new_tokens=self.max_new,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=self.tokenizer.pad_token_id,
            return_dict_in_generate=True,
            output_scores=True,
            use_cache=False,
        )
        gen_ids = out.sequences[0, input_ids.shape[1]:]
        text    = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
        scores  = torch.stack(out.scores, dim=0)
        lp_all  = F.log_softmax(scores, dim=-1)
        lp      = lp_all[torch.arange(len(gen_ids)), gen_ids]
        return text, gen_ids, lp

    @torch.no_grad()
    def _ref_log_probs(self, full_ids, prompt_len):
        out     = self.ref_model(full_ids)
        gen_len = full_ids.shape[1] - prompt_len
        logits  = out.logits[0, prompt_len - 1: prompt_len - 1 + gen_len]
        lp      = F.log_softmax(logits, dim=-1)
        gen_ids = full_ids[0, prompt_len:]
        return lp[torch.arange(len(gen_ids)), gen_ids]

    def _ppo_loss(self, full_ids, prompt_len, old_lp, advantage):
        gen_ids  = full_ids[0, prompt_len:]
        out      = self.model(full_ids)
        gen_len  = len(gen_ids)
        logits   = out.logits[0, prompt_len - 1: prompt_len - 1 + gen_len]
        new_lp   = F.log_softmax(logits, dim=-1)[torch.arange(gen_len), gen_ids]
        ratio    = torch.exp(new_lp - old_lp.to(DEVICE))
        A        = torch.tensor(advantage, dtype=torch.float32, device=DEVICE)
        surr     = torch.min(ratio * A, torch.clamp(ratio, 1 - self.eps, 1 + self.eps) * A)
        ref_lp   = self._ref_log_probs(full_ids, prompt_len)
        kl       = (new_lp - ref_lp).mean()
        loss     = -surr.mean() + self.beta * kl
        return loss, (-surr.mean()).item(), (self.beta * kl).item()

    def step(self, batch: List[Dict], step_idx: int, alpha: Optional[float] = None) -> Dict:
        self.model.train()
        alpha = alpha if alpha is not None else self.alpha
        self.optimizer.zero_grad()
        total_loss = total_pl = total_kl = total_reward = 0.0
        total_correct = total_resp = 0
        all_lens = []

        for item in batch:
            question = item['question']
            gt       = item['final_answer']
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': question},
            ]
            prompt_text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            enc = self.tokenizer(
                prompt_text, return_tensors='pt', truncation=True,
                max_length=MAX_SEQ_LENGTH // 2
            ).to(DEVICE)
            prompt_len = enc['input_ids'].shape[1]

            texts, gen_ids_list, old_lps = [], [], []
            for _ in range(self.n):
                text, gen_ids, lp = self._generate_one(enc['input_ids'])
                texts.append(text)
                gen_ids_list.append(gen_ids)
                old_lps.append(lp)
                all_lens.append(len(text.split()))

            rewards, correctness = self.reward.compute_rewards(texts, gt, alpha=alpha)
            advantages = RLOOAdvantage.compute(rewards)

            n_total = len(batch) * self.n
            for gen_ids, old_lp, adv in zip(gen_ids_list, old_lps, advantages):
                full_ids = torch.cat([enc['input_ids'][0], gen_ids.to(DEVICE)]).unsqueeze(0)
                loss, pl, kl = self._ppo_loss(full_ids, prompt_len, old_lp, adv)
                (loss / n_total).backward()
                total_loss += loss.item()
                total_pl   += pl
                total_kl   += kl

            total_reward  += sum(rewards)
            total_correct += sum(correctness)
            total_resp    += len(texts)

        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        metrics = {
            'loss':        total_loss  / max(total_resp, 1),
            'policy_loss': total_pl    / max(total_resp, 1),
            'kl':          total_kl    / max(total_resp, 1),
            'reward':      total_reward / max(total_resp, 1),
            'accuracy':    total_correct / max(total_resp, 1),
            'mean_tokens': np.mean(all_lens) if all_lens else 0.0,
        }
        for k, v in metrics.items():
            self.history[k].append(v)
        return metrics


print('PPOTrainerRLOO defined')


---
## Extension 1 — Dynamic Alpha Curriculum (DAC)

Adapts the length-penalty strength `alpha` per problem:
- **Hard problems** (low pass@1) → lower alpha → accuracy is protected
- **Easy problems** (high pass@1) → higher alpha → stronger compression


In [ ]:
class DynamicAlphaCurriculum:
    """
    alpha(x, k) = alpha_base + alpha_range . h(x) . s(k)
    h(x) = 1 - pass@1 rate (online hardness estimate)
    s(k) = cosine curriculum schedule in [0,1]
    Easy problems -> high alpha -> strong compression
    Hard problems -> low  alpha -> protect accuracy
    """

    def __init__(self, alpha_base=0.1, alpha_range=0.15, total_steps=100):
        self.alpha_base  = alpha_base
        self.alpha_range = alpha_range
        self.total_steps = total_steps
        self._attempts   = {}
        self._correct    = {}

    def update(self, question: str, correct: bool):
        k = question[:60]
        self._attempts[k] = self._attempts.get(k, 0) + 1
        if correct:
            self._correct[k] = self._correct.get(k, 0) + 1

    def hardness(self, question: str) -> float:
        k     = question[:60]
        n     = self._attempts.get(k, 0)
        c     = self._correct.get(k, 0)
        pass1 = c / n if n > 0 else 0.5
        return 1.0 - pass1

    def schedule(self, step: int) -> float:
        return 0.5 * (1 - math.cos(math.pi * step / max(self.total_steps, 1)))

    def get_alpha(self, question: str, step: int) -> float:
        return float(np.clip(
            self.alpha_base + self.alpha_range * self.hardness(question) * self.schedule(step),
            0.0, 0.5
        ))


print('DynamicAlphaCurriculum (DAC) defined')


---
## Extension 2 — Multi-Objective Reward Shaping (MORS)

Combines four weighted sub-rewards:
| Component | Weight | Measures |
|-----------|--------|----------|
| R_corr | 0.70 | Correctness |
| R_len  | 0.15 | Brevity among correct responses |
| R_qual | 0.10 | Semantic faithfulness (cosine sim) |
| R_fmt  | 0.05 | Structured answer format |


In [ ]:
# Install sentence-transformers if not already present
try:
    from sentence_transformers import SentenceTransformer, util as st_util
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'sentence-transformers'], check=True)
    from sentence_transformers import SentenceTransformer, util as st_util


class MORSReward:
    """
    R_total = 0.70.R_corr + 0.15.R_len + 0.10.R_qual + 0.05.R_fmt
    """
    W = {'corr': 0.70, 'len': 0.15, 'qual': 0.10, 'fmt': 0.05}

    def __init__(self):
        self._embedder = None

    @property
    def embedder(self):
        if self._embedder is None:
            self._embedder = SentenceTransformer('all-MiniLM-L6-v2')
        return self._embedder

    def r_corr(self, output, gt):       return 1.0 if is_correct(output, gt) else 0.0

    def r_len(self, output, outputs, gt):
        correct = [o for o in outputs if is_correct(o, gt)]
        if not correct or not is_correct(output, gt): return 0.0
        mn = min(len(o.split()) for o in correct)
        mx = max(len(o.split()) for o in correct)
        if mx == mn: return 1.0
        return 1.0 - (len(output.split()) - mn) / (mx - mn + 1e-8)

    def r_qual(self, output, reference):
        e_out = self.embedder.encode(output,    convert_to_tensor=True)
        e_ref = self.embedder.encode(reference, convert_to_tensor=True)
        return max(0.0, float(st_util.cos_sim(e_out, e_ref).item()))

    def r_fmt(self, output):
        lower = output.lower()
        s  = 0.5 if any(k in lower for k in ['step', 'first', 'then', 'finally']) else 0.0
        s += 0.5 if any(k in lower for k in ['####', '<stop>', 'answer:']) else 0.0
        return s

    def compute(self, output, outputs, gt, reference=''):
        rc = self.r_corr(output, gt)
        rl = self.r_len(output, outputs, gt)
        rq = self.r_qual(output, reference or gt)
        rf = self.r_fmt(output)
        total = self.W['corr']*rc + self.W['len']*rl + self.W['qual']*rq + self.W['fmt']*rf
        return total, {'R_corr': rc, 'R_len': rl, 'R_qual': rq, 'R_fmt': rf, 'R_total': total}


print('MORSReward defined')


---
## Stage 2B — Reinforcement Learning: PPO + RLOO + Length Penalty

In [ ]:
# Build RL prompt list from the loaded conversations
import re

rl_prompts = []
for conv in conversations:
    msgs     = conv.get('messages', [])
    user_msg = next((m['content'] for m in msgs if m['role'] == 'user'),  None)
    asst_msg = next((m['content'] for m in msgs if m['role'] == 'assistant'), None)
    if user_msg and asst_msg:
        m = re.match(r'.*####\s*([\d\.\,\-]+)', asst_msg, re.DOTALL)
        final_ans = m.group(1).strip().replace(',', '') if m else asst_msg.strip()
        rl_prompts.append({
            'question':     user_msg.strip(),
            'answer':       asst_msg.strip(),
            'final_answer': final_ans,
        })

print(f"RL prompts ready: {len(rl_prompts)}")
if rl_prompts:
    print(f'  Sample Q: {rl_prompts[0]["question"][:70]}...')
    print(f'  Answer:   {rl_prompts[0]["final_answer"]}')


In [ ]:
from transformers import AutoTokenizer

SFT_OUTPUT = './fine_tuned_phi_4_mini'

RL_CONFIG = {
    'n_steps':        100,
    'batch_size':     4,
    'n_samples':      8,
    'alpha':          0.1,
    'ppo_clip':       0.2,
    'kl_beta':        0.02,
    'lr':             1e-5,
    'max_new_tokens': 256,
    'use_dac':        True,
    'log_every':      10,
}

print('Loading SFT checkpoint for RL fine-tuning...')

rl_tokenizer = AutoTokenizer.from_pretrained(SFT_OUTPUT, trust_remote_code=True)
if rl_tokenizer.pad_token is None:
    rl_tokenizer.pad_token = rl_tokenizer.eos_token
rl_tokenizer.pad_token_id = rl_tokenizer.eos_token_id

rl_policy, _ = FastLanguageModel.from_pretrained(
    model_name=SFT_OUTPUT,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
rl_policy.resize_token_embeddings(len(rl_tokenizer))
rl_policy = FastLanguageModel.get_peft_model(
    rl_policy,
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'fc1', 'fc2'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
FastLanguageModel.for_training(rl_policy)
rl_policy.config.pad_token_id = rl_tokenizer.pad_token_id
rl_policy.config.eos_token_id = rl_tokenizer.eos_token_id
rl_policy.config.use_cache = False

print('Loading frozen reference model...')
rl_ref, _ = FastLanguageModel.from_pretrained(
    model_name=SFT_OUTPUT,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
rl_ref.resize_token_embeddings(len(rl_tokenizer))
rl_ref.eval()
for p in rl_ref.parameters():
    p.requires_grad_(False)
rl_ref.config.pad_token_id = rl_tokenizer.pad_token_id
rl_ref.config.eos_token_id = rl_tokenizer.eos_token_id
rl_ref.config.use_cache = False

reward_module = RewardModule(alpha=RL_CONFIG['alpha'])

dac = DynamicAlphaCurriculum(
    alpha_base=RL_CONFIG['alpha'],
    alpha_range=0.15,
    total_steps=RL_CONFIG['n_steps'],
) if RL_CONFIG['use_dac'] else None

ppo_trainer = PPOTrainerRLOO(
    model=rl_policy,
    ref_model=rl_ref,
    tokenizer=rl_tokenizer,
    reward_module=reward_module,
    alpha=RL_CONFIG['alpha'],
    ppo_clip=RL_CONFIG['ppo_clip'],
    kl_beta=RL_CONFIG['kl_beta'],
    lr=RL_CONFIG['lr'],
    n_samples=RL_CONFIG['n_samples'],
    max_new_tokens=RL_CONFIG['max_new_tokens'],
)

print(f"Starting RL training: {RL_CONFIG['n_steps']} steps")
print(f"  {RL_CONFIG['batch_size']} prompts x {RL_CONFIG['n_samples']} responses each")
print(f"  DAC: {RL_CONFIG['use_dac']}")
print('=' * 60)

for step in range(RL_CONFIG['n_steps']):
    batch = random.sample(rl_prompts, min(RL_CONFIG['batch_size'], len(rl_prompts)))

    step_alpha = None
    if dac:
        alphas = [dac.get_alpha(item['question'], step) for item in batch]
        step_alpha = float(np.mean(alphas))

    metrics = ppo_trainer.step(batch, step_idx=step, alpha=step_alpha)

    if dac:
        for item in batch:
            dac.update(item['question'], metrics['accuracy'] > 0.5)

    if step % RL_CONFIG['log_every'] == 0 or step == RL_CONFIG['n_steps'] - 1:
        dac_str = f' | alpha={step_alpha:.3f}' if step_alpha is not None else ''
        loss_str = f"  Step {step:3d}/{RL_CONFIG['n_steps']}"
        loss_str += f" | Loss={metrics['loss']:.4f}"
        loss_str += f" | Reward={metrics['reward']:.3f}"
        loss_str += f" | Acc={metrics['accuracy']*100:.1f}%"
        loss_str += f" | Tokens={metrics['mean_tokens']:.0f}"
        loss_str += f" | KL={metrics['kl']:.4f}"
        loss_str += dac_str
        print(loss_str)

print('=' * 60)
rl_policy.save_pretrained('./rl_efficient_reasoning')
print('RL training complete -> ./rl_efficient_reasoning')


---
## Stage: Automatic Evaluation — 5 Metrics

| Metric | Target |
|--------|--------|
| Exact Match Accuracy | >= 78% |
| Token Reduction | >= 40% vs baseline |
| Faithfulness Score | >= 0.82 |
| Response Consistency (k=5) | >= 0.85 |
| Latency Reduction | ~2-3x |


In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

try:
    from sentence_transformers import SentenceTransformer, util as st_util
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'sentence-transformers'], check=True)
    from sentence_transformers import SentenceTransformer, util as st_util


class EvaluationSuite:
    """Automatic evaluation suite for the efficient reasoning model."""

    def __init__(self):
        print('Loading sentence-transformer for faithfulness metric...')
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

    def exact_match_accuracy(self, outputs, ground_truths):
        correct = sum(is_correct(o, g) for o, g in zip(outputs, ground_truths))
        acc     = correct / max(len(outputs), 1)
        return {'metric': 'Exact Match Accuracy', 'value': acc, 'target': 0.78,
                'pass': acc >= 0.78, 'correct': correct, 'total': len(outputs)}

    def token_reduction(self, efficient, baseline):
        mean_eff  = np.mean([len(o.split()) for o in efficient])
        mean_base = np.mean([len(o.split()) for o in baseline])
        red = 1.0 - mean_eff / (mean_base + 1e-8)
        return {'metric': 'Token Reduction', 'value': red, 'target': 0.40,
                'pass': red >= 0.40, 'reduction_pct': red * 100,
                'mean_efficient_tokens': mean_eff, 'mean_baseline_tokens': mean_base}

    def faithfulness_score(self, outputs, references):
        scores = []
        for out, ref in zip(outputs, references):
            e_out = self.embedder.encode(out, convert_to_tensor=True)
            e_ref = self.embedder.encode(ref, convert_to_tensor=True)
            scores.append(max(0.0, float(st_util.cos_sim(e_out, e_ref).item())))
        mean_f = np.mean(scores)
        return {'metric': 'Faithfulness Score', 'value': mean_f, 'target': 0.82,
                'pass': mean_f >= 0.82}

    def response_consistency(self, model, tokenizer, questions, k=5):
        rates = []
        for q in questions:
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': q},
            ]
            text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
            answers = []
            for _ in range(k):
                with torch.no_grad():
                    out = model.generate(
                        **inputs, max_new_tokens=200,
                        do_sample=True, temperature=0.7, use_cache=True,
                        pad_token_id=tokenizer.pad_token_id,
                    )
                gen = tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
                ans = extract_answer(gen)
                if ans:
                    answers.append(ans)
            if len(answers) >= 2:
                pairs = [(a, b) for i, a in enumerate(answers)
                         for j, b in enumerate(answers) if i < j]
                try:
                    agree = sum(abs(float(a) - float(b)) < 1e-3 for a, b in pairs)
                    rates.append(agree / len(pairs))
                except (ValueError, TypeError):
                    rates.append(0.0)
            else:
                rates.append(0.0)
        mean_c = np.mean(rates) if rates else 0.0
        return {'metric': 'Response Consistency', 'value': mean_c, 'target': 0.85,
                'pass': mean_c >= 0.85, 'k': k}

    def latency_reduction(self, efficient, baseline):
        ratio = np.mean([len(o.split()) for o in baseline]) / (
            np.mean([len(o.split()) for o in efficient]) + 1e-8
        )
        return {'metric': 'Latency Reduction', 'value': ratio, 'target': 2.0,
                'pass': ratio >= 2.0, 'speedup_factor': ratio}

    def evaluate_all(self, model, tokenizer, test_prompts, n=50, k=5):
        FastLanguageModel.for_inference(model)
        test_prompts = test_prompts[:n]
        print(f"Generating {n} efficient + baseline outputs...")
        efficient_outputs, baseline_outputs = [], []
        for item in test_prompts:
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': item['question']},
            ]
            text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
            pl = inputs['input_ids'].shape[1]
            with torch.no_grad():
                out_eff  = model.generate(**inputs, max_new_tokens=256,
                    do_sample=False, use_cache=True, pad_token_id=tokenizer.pad_token_id)
                out_base = model.generate(**inputs, max_new_tokens=512,
                    do_sample=True, temperature=1.0, use_cache=True, pad_token_id=tokenizer.pad_token_id)
            efficient_outputs.append(tokenizer.decode(out_eff[0, pl:],  skip_special_tokens=True))
            baseline_outputs.append( tokenizer.decode(out_base[0, pl:], skip_special_tokens=True))

        gt_list  = [item['final_answer'] for item in test_prompts]
        ref_list = [item['answer']       for item in test_prompts]

        print('Computing metrics...')
        results = {}
        results['accuracy']        = self.exact_match_accuracy(efficient_outputs, gt_list)
        results['token_reduction'] = self.token_reduction(efficient_outputs, baseline_outputs)
        results['faithfulness']    = self.faithfulness_score(efficient_outputs, ref_list)
        results['consistency']     = self.response_consistency(
            model, tokenizer, [item['question'] for item in test_prompts[:20]], k=k
        )
        results['latency']         = self.latency_reduction(efficient_outputs, baseline_outputs)

        print('\n' + '=' * 62)
        print(f'{"METRIC":<28} {"ACHIEVED":>10} {"TARGET":>10} {"PASS":>8}')
        print('-' * 62)
        FMT = {
            'accuracy':        lambda v, t: (f'{v*100:.1f}%', f'>={t*100:.0f}%'),
            'token_reduction': lambda v, t: (f'{v*100:.1f}%', f'>={t*100:.0f}%'),
            'faithfulness':    lambda v, t: (f'{v:.4f}',      f'>={t:.2f}'),
            'consistency':     lambda v, t: (f'{v:.4f}',      f'>={t:.2f}'),
            'latency':         lambda v, t: (f'{v:.2f}x',     f'>={t:.0f}x'),
        }
        for key, res in results.items():
            vs, ts = FMT[key](res['value'], res['target'])
            flag   = 'PASS' if res['pass'] else 'FAIL'
            print(f"{res['metric']:<28} {vs:>10} {ts:>10} {flag:>8}")
        print('=' * 62)
        tr = results['token_reduction']
        print('\nToken summary:')
        print(f'  Efficient: {tr["mean_efficient_tokens"]:.0f} tokens/response')
        print(f'  Baseline:  {tr["mean_baseline_tokens"]:.0f} tokens/response')
        print(f'  Reduction: {tr["reduction_pct"]:.1f}%')
        print(f'  Speedup:   {results["latency"]["speedup_factor"]:.2f}x')
        return results


print('EvaluationSuite defined')


In [ ]:
eval_suite   = EvaluationSuite()
eval_results = eval_suite.evaluate_all(
    model=rl_policy,
    tokenizer=rl_tokenizer,
    test_prompts=rl_prompts,
    n=50,
    k=5,
)


---
## Visualisations — Training Curves & Eval Results

In [ ]:
import matplotlib.pyplot as plt


def plot_training_curves(trainer):
    h      = trainer.history
    keys   = ['loss', 'reward', 'accuracy', 'kl', 'mean_tokens', 'policy_loss']
    labels = ['Total Loss', 'Mean Reward', 'Accuracy', 'KL Divergence', 'Mean Tokens', 'Policy Loss']
    colors = ['#E74C3C', '#2ECC71', '#3498DB', '#9B59B6', '#F39C12', '#1ABC9C']
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle('RL Training - PPO + RLOO + Length Penalty', fontsize=14, fontweight='bold')
    for ax, key, label, color in zip(axes.flatten(), keys, labels, colors):
        vals = h.get(key, [])
        if not vals:
            ax.set_visible(False)
            continue
        ax.plot(vals, color=color, linewidth=1.5, alpha=0.4)
        if len(vals) > 5:
            w  = max(1, len(vals) // 10)
            sm = np.convolve(vals, np.ones(w) / w, mode='valid')
            ax.plot(range(len(sm)), sm, color=color, linewidth=2.5)
        ax.set_title(label, fontweight='bold')
        ax.set_xlabel('Step')
        ax.grid(True, alpha=0.3)
        ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig('rl_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_eval_results(eval_results):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Evaluation Results vs Targets', fontsize=13, fontweight='bold')
    names, achieved, targets = [], [], []
    for key, res in eval_results.items():
        names.append(res['metric'].replace(' ', '\n'))
        v, t = res['value'], res['target']
        if key in ('accuracy', 'token_reduction'): v, t = v * 100, t * 100
        achieved.append(v)
        targets.append(t)
    x = np.arange(len(names))
    w = 0.35
    b1 = ax1.bar(x - w/2, achieved, w, label='Achieved', color='#2ECC71', alpha=0.85)
    ax1.bar(x + w/2, targets, w, label='Target', color='#3498DB', alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels(names, fontsize=7)
    ax1.legend()
    ax1.set_title('Metrics vs Targets')
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.spines[['top', 'right']].set_visible(False)
    for bar in b1:
        h = bar.get_height()
        ax1.annotate(f'{h:.1f}', xy=(bar.get_x() + bar.get_width() / 2, h),
                     xytext=(0, 2), textcoords='offset points', ha='center', fontsize=7)
    passed = sum(1 for r in eval_results.values() if r['pass'])
    failed = len(eval_results) - passed
    ax2.pie([passed, failed],
            labels=[f'Passed ({passed})', f'Failed ({failed})'],
            colors=['#2ECC71', '#E74C3C'], autopct='%1.0f%%', startangle=90)
    ax2.set_title('Pass / Fail')
    plt.tight_layout()
    plt.savefig('eval_benchmark.png', dpi=150, bbox_inches='tight')
    plt.show()


print('Visualisation functions defined')


In [ ]:
plot_training_curves(ppo_trainer)
plot_eval_results(eval_results)


---
## Stage 5 · Inference Demo

In [ ]:
import time
FastLanguageModel.for_inference(rl_policy)

DEMO_QUESTIONS = [
    'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
    'Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?',
    'Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?',
]

print('=' * 60)
print('EFFICIENT REASONING DEMO - Phi-4-Mini (RL-trained)')
print('=' * 60)
for q in DEMO_QUESTIONS:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': q},
    ]
    text   = rl_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = rl_tokenizer(text, return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = rl_policy.generate(
            **inputs, max_new_tokens=256, do_sample=False, use_cache=True,
            pad_token_id=rl_tokenizer.pad_token_id,
        )
    elapsed  = time.time() - t0
    response = rl_tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    tokens   = len(response.split())
    answer   = extract_answer(response)
    print(f"\nQ: {q}")
    print(f"A: {response}")
    print(f"   Tokens: {tokens} | Latency: {elapsed:.2f}s | Answer: {answer}")
    print('-' * 60)


---
## Summary Report

In [ ]:
import json as _json

summary = {}
for k, v in eval_results.items():
    summary[k] = {
        ky: float(va) if isinstance(va, (float, np.floating)) else va
        for ky, va in v.items() if not isinstance(va, list)
    }
summary['model']     = MODEL_ID
summary['rl_config'] = RL_CONFIG
summary['timestamp'] = time.strftime('%Y-%m-%d %H:%M:%S')

with open('summary_report.json', 'w') as f:
    _json.dump(summary, f, indent=2)

print('Summary report saved -> summary_report.json')
print(_json.dumps({k: v for k, v in summary.items() if k not in ('rl_config',)}, indent=2))


## Export Options

Run the cells below to export your model to different formats.


In [ ]:
#@title Export to GGUF (for Ollama, LM Studio) - Optional
#@markdown Check the box to enable export. **Adds ~5-10 min.**
run_gguf_export = False  #@param {type:"boolean"}

if run_gguf_export:
    GGUF_OUTPUT = "./fine_tuned_phi_4_mini_gguf"
    QUANT_METHOD = "q4_k_m"  # Good balance of size/quality

    print(f"Exporting to GGUF ({QUANT_METHOD})...")
    model.save_pretrained_gguf(GGUF_OUTPUT, tokenizer, quantization_method=QUANT_METHOD)
    print(f"GGUF model saved to: {GGUF_OUTPUT}")

    import shutil
    shutil.make_archive(GGUF_OUTPUT, 'zip', GGUF_OUTPUT)
    print(f"\nDownload: {GGUF_OUTPUT}.zip")
else:
    print("Skipped GGUF export. Check the box above and re-run if needed.")


Skipped GGUF export. Check the box above and re-run if needed.


In [ ]:
#@title Export Merged Model (for HuggingFace) - Optional
#@markdown Check the box to enable export. **Adds ~3-5 min.**
run_merged_export = False  #@param {type:"boolean"}

if run_merged_export:
    MERGED_OUTPUT = "./fine_tuned_phi_4_mini_merged"

    print("Exporting 16-bit merged model...")
    model.save_pretrained_merged(MERGED_OUTPUT + "_16bit", tokenizer, save_method="merged_16bit")
    print(f"Saved to: {MERGED_OUTPUT}_16bit")
else:
    print("Skipped merged export. Check the box above and re-run if needed.")


Skipped merged export. Check the box above and re-run if needed.


## Download Your Model

1. **Click the folder icon** in the left sidebar
2. **Navigate to the output folder** (e.g., `fine_tuned_phi_4_mini`)
3. **Right-click** on the file/folder > **Download**

### Using Your Model

**With Unsloth (recommended):**
```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./fine_tuned_phi_4_mini",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```

**With HuggingFace (after merging):**
```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("./fine_tuned_phi_4_mini_merged_16bit")
tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_phi_4_mini_merged_16bit")
```

**With Ollama (after GGUF export):**
```bash
# Create Modelfile
echo 'FROM ./fine_tuned_phi_4_mini_gguf/model.gguf' > Modelfile
ollama create my-model -f Modelfile
ollama run my-model
```

---
*Happy fine-tuning!*
